<a href="https://colab.research.google.com/github/SanaKamranButt/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanaKamranButt/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents the daily search performance of one content item for one client.

For this assignment, I use data from March 2026 because it is a mid-panel month. The goal is to build features from historical search performance to identify pages that may require a content refresh.

In [17]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste HF Token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

fact = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

grain = con.sql(f"""
SELECT
COUNT(*) AS rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {fact}
""").df()

grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- report_date
- client_hash_id

Label:
Whether a content page experiences declining search performance and should be prioritized for review, based on observed historical search data.

Context:
client_hash_id
content_hash_id
report_date

Excluded:
URL, page title, and client-identifying information because they are not needed for the model and should remain private.

In [18]:
con.sql(f"""
SELECT *
FROM {fact}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

The following queries verify the row count, date range, and feature availability for the selected month.

In [19]:
# Query 1 - Row count

print("Rows:")
print(con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM {fact}
""").df())

# Query 2 - Date window

print("\nDate Range:")
print(con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {fact}
""").df())

# Query 3 - Feature availability

print("\nFeature availability:")
print(con.sql(f"""
SELECT
COUNT(*) AS total_rows,
COUNT(gsc_impressions) AS impressions_available,
COUNT(gsc_clicks) AS clicks_available,
COUNT(gsc_avg_position) AS position_available
FROM {fact}
""").df())

Rows:
   total_rows
0     9841378

Date Range:
  start_date   end_date
0 2026-03-01 2026-03-31

Feature availability:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  impressions_available  clicks_available  position_available
0     9841378                9841378           9841378             3611061


## 4. Data limits

This dataset cannot explain why search performance changes. It contains observed historical search data only. Different clients have different history lengths, and some earlier records contain only Google Search Console data. The model provides decision-support based on observed patterns and does not predict Google's algorithm or prove cause and effect.The dataset cannot capture external factors such as Google algorithm updates, seasonality, or content changes outside the recorded search metrics.

In [20]:
print("Data contract completed successfully.")

Data contract completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.